# Ch.1 — GPU Architecture Fundamentals

**No GPU required.** Every cell in this notebook runs on any laptop using NumPy, Pandas, and Matplotlib. You are building calculators and visualisers — the same tools a Platform Engineer uses to reason about hardware *before* spending money on it.

**What this notebook builds:**
1. A GPU spec database — compare cards side-by-side on the numbers that actually matter
2. A Roofline Model visualiser — see exactly where any operation sits relative to a GPU's limits
3. An arithmetic intensity calculator — compute the FLOP/byte ratio for every major LLM operation
4. A decode step simulator — trace one token generation through the hardware at the byte level
5. A batching curve — watch arithmetic intensity rise as batch size grows
6. An InferenceBase decision tool — which GPU should the startup buy?

In [ ]:
# TODO: Implement this cell


## 1 · GPU Spec Database

The five numbers that matter for AI workloads — everything else is marketing.

| Spec | Why it matters |
|---|---|
| `bf16_tflops` | Peak compute for LLM training/inference (use this, not FP32) |
| `bandwidth_tbs` | HBM bandwidth in TB/s — the real bottleneck for decode |
| `vram_gb` | Maximum model + KV cache you can fit |
| `nvlink_bw_gbs` | Intra-node GPU-to-GPU bandwidth (0 = PCIe only) |
| `tdp_w` | Watts — cooling, rack density, electricity cost |

Run the cell below to load the database, then explore `gpu_db`.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 2 · The Roofline Model

The Roofline Model answers one question: *for a given GPU, is my operation limited by compute or by memory?*

$$\text{Achievable TFLOP/s} = \min\!\left(\text{Peak TFLOP/s},\;\; I \times \text{Bandwidth (TB/s)}\right)$$

Where $I$ is **arithmetic intensity** in FLOP/byte:

$$I = \frac{\text{FLOPs performed}}{\text{Bytes read from HBM}}$$

- If $I < I_\text{ridge}$: compute units idle waiting for data → **memory-bound**
- If $I > I_\text{ridge}$: memory system is idle → **compute-bound**
- $I_\text{ridge} = \text{Peak TFLOP/s} \div \text{Bandwidth (TB/s)}$

The `plot_roofline()` function below works for any GPU in `gpu_db`. Pass a list of GPU names and optionally a list of `(label, intensity)` operation points to overlay.

In [ ]:
# TODO: Implement this cell


## 3 · Arithmetic Intensity of LLM Operations

Now let's calculate where real LLM operations land on that roofline.

For a matrix multiply $[B \times M] \times [M \times N]$:

$$\text{FLOPs} = 2 \cdot B \cdot M \cdot N$$
$$\text{Bytes} = (B \cdot M + M \cdot N + B \cdot N) \times \text{bytes per element}$$
$$I = \frac{2BMN}{(BM + MN + BN) \times \text{bpe}}$$

When $B = 1$ (single user, token-by-token decode), the weight matrix $[M \times N]$ completely dominates the byte count. The intensity collapses toward $\approx 1 / \text{bpe}$ — nearly zero relative to the ridge point. This is the root cause of why single-user LLM inference is so GPU-inefficient.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 4 · Decode Step Simulator — One Token at the Hardware Level

Let's trace exactly what happens during a single decode step for Llama-3-8B: FLOPs performed, bytes moved from HBM, and the resulting minimum latency bounded by memory bandwidth — not compute.

The simulator accounts for every operation in every layer: QKV projections, attention score computation, softmax + weighted sum, output projection, and both FFN linear layers. The LM head (vocab projection) is included once.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 5 · The Batching Curve — Why Batching Transforms Efficiency

When processing $B$ tokens simultaneously instead of 1, the weight matrices are still the same size (loaded once from HBM), but you perform $B\times$ more FLOPs on them. Arithmetic intensity rises linearly with batch size — until you cross the ridge point and become compute-bound.

$$I(B) \xrightarrow{B \gg 1} \frac{2B}{\text{bpe}}$$

This is the fundamental reason why **continuous batching** (Ch.5) is the single most impactful inference optimisation: it raises the effective batch size from ~1 toward 32–128 without adding latency per user.

In [ ]:
# TODO: Implement this cell


## 6 · InferenceBase Decision Tool

The Platform Engineer needs to answer: **which GPU should InferenceBase buy to serve Llama-3-8B at 5,000 tokens/second total (100 concurrent users × 50 tok/s each), within a $15,000/month cloud budget?**

We model this using only bandwidth arithmetic and published cloud pricing — no benchmarks, no live GPU required.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 7 · Sensitivity Analysis — How Model Size Changes Everything

How do the key metrics change as the model scales from 1B to 70B parameters? The VRAM footprint is the hard constraint. Arithmetic intensity at batch=1 barely changes with model size — all models are equally memory-bound per token. But the bandwidth requirement (HBM bytes per decode step) scales directly with parameter count.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Exercises

Work through these before moving to Ch.2. Each one extends the calculators above.

**Exercise 1** — Add your own GPU to `gpu_specs` in section 1 and re-run all cells. Where does it sit on the roofline?

**Exercise 2** — FP8 inference on H100. Re-run `decode_step_analysis` with `bpe=0.5`. Does the H100's higher ridge point make FP8 decode compute-bound before BF16?

**Exercise 3** — Longer context. Re-run `decode_step_analysis` with `seq_len=4096`. Does quadrupling the context significantly change arithmetic intensity?

**Exercise 4** — The 70B upgrade. Model what happens if InferenceBase upgrades to Llama-3-70B. Does any single GPU fit the model in BF16? What is the minimum GPU count and cost?

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell
